In [1]:
#Struktur KB
facts = set()
rules = []
derivation_trace = {}

def add_fact(predicate, *args):
    facts.add((predicate, *args))

def add_rule(conditions, conclusion):
    rules.append((conditions, conclusion))

In [2]:
#Fakta KB
add_rule([("hypertensive", "X")], ("has_diabetes", "X")) #High BP (BP>140)
add_rule([("age_over_45", "X")], ("has_diabetes", "X")) #Age (>45)
add_rule([("high_bmi", "X")], ("has_diabetes", "X")) #BMI (>30)
add_rule([("smoker", "X")], ("has_diabetes", "X"))  #Batang rokok (>100)
add_rule([("stressed", "X")], ("has_diabetes", "X")) # Stressed Yes or no (0-30) (>15 yes)
add_rule([("low_income", "X")], ("has_diabetes", "X")) #skala 1-8 
add_rule([("female", "X")], ("has_diabetes", "X"))  #Male or Female

In [3]:
#Rule FOL
ind_list = [
    "hypertensive", "age_over_45", "high_bmi", "smoker",
    "stressed", "low_income", "female"
]

for ind in ind_list:
    add_rule(
        conditions=[(ind, "X")],
        conclusion=("indicator_met", "X", ind)
    )

In [4]:
#Unify, Substitute, Forward Chain
def unify(pattern, fact):
    if pattern[0] != fact[0]: return None
    if len(pattern) != len(fact): return None

    bind = {}
    for p, f in zip(pattern, fact):
        if p == "X":
            bind["X"] = f
        elif p != f:
            return None
    return bind


def substitute(expr, bind):
    return tuple(bind.get(arg, arg) for arg in expr)


def forward_chain():
    added = True
    while added:
        added = False

        for conditions, conclusion in rules:
            match_list = []

            for cond in conditions:
                matches = []
                for f in facts:
                    b = unify(cond, f)
                    if b is not None:
                        matches.append(b)
                if not matches:
                    break
                match_list.append(matches)

            else:
                for bind in match_list[0]:
                    new_conclusion = substitute(conclusion, bind)
                    new_conditions = [
                        substitute(cond, bind) for cond in conditions
                    ]
                    if new_conclusion not in facts:
                        facts.add(new_conclusion)
                        derivation_trace[new_conclusion] = new_conditions
                        added = True

In [5]:
#Generate Fact
def generate_facts_from_input(data):
    name = data["name"]

    # BP > 140 = hypertensive
    if data["BP"] > 140:
        add_fact("hypertensive", name)

    # Age > 45
    if data["age"] > 45:
        add_fact("age_over_45", name)

    # BMI > 30
    if data["BMI"] > 30:
        add_fact("high_bmi", name)

    # Cigarettes > 100 = smoker
    if data["cigarettes"] > 100:
        add_fact("smoker", name)

    # Stress = 15 (0–30 scale)
    if data["stress"] > 15:
        add_fact("stressed", name)

    # Income scale 1–3 = low income
    income = data["income_scale"]
    if income in [1,2,3]:
        add_fact("low_income", name)

    # Gender: female = indicator
    if data["gender"].lower() == "female":
        add_fact("female", name)

In [6]:
#Laporan diagnosis
def get_report(person):
    forward_chain()

    indicators = []
    for f in facts:
        if f[0] == "indicator_met" and f[1] == person:
            indicators.append(f[2])

    diagnosis = "no diabetes"
    if ("has_diabetes", person) in facts:
        diagnosis = "has diabetes"

    print(f"nama: {person}")
    print(f"indicators met: {len(indicators)}/{len(ind_list)}")
    print("indicators:")
    for i, ind in enumerate(indicators, 1):
        print(f"{i}. {ind}")
    print()
    print("diagnosis:", diagnosis)
    print("\n" + "-"*40 + "\n")

In [7]:
#Data contoh
person1 = {
    "name": "alice",
    "BP": 160,
    "age": 52,
    "BMI": 33,
    "cigarettes": 0,
    "stress": 20,
    "income_scale": 2,
    "gender": "female"
}

person2 = {
    "name": "bob",
    "BP": 120,
    "age": 28,
    "BMI": 22,
    "cigarettes": 50,
    "stress": 10,
    "income_scale": 7,
    "gender": "male"
}


In [8]:
#Hasil query
generate_facts_from_input(person1)
generate_facts_from_input(person2)

get_report("alice")
get_report("bob")

nama: alice
indicators met: 6/7
indicators:
1. low_income
2. high_bmi
3. stressed
4. hypertensive
5. age_over_45
6. female

diagnosis: has diabetes

----------------------------------------

nama: bob
indicators met: 0/7
indicators:

diagnosis: no diabetes

----------------------------------------

